In [1]:
import pandas as pd
import spacy


/home/nmi/projects/grassroot_drones/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1112: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/nmi/projects/grassroot_drones/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1170: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


In [4]:
!!pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl


['Collecting en-core-web-sm==3.8.0',
 '  Downloading en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)',
 '\x1b[?25l     \x1b━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\x1b \x1b0.0/12.8 MB\x1b \x1b?\x1b eta \x1b-:--:--\x1b',
 '\x1b[2K     \x1b━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\x1b \x1b0.0/12.8 MB\x1b \x1b?\x1b eta \x1b-:--:--\x1b',
 '\x1b[2K     \x1b━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\x1b \x1b0.0/12.8 MB\x1b \x1b?\x1b eta \x1b-:--:--\x1b',
 '\x1b[2K     \x1b━\x1b\x1b╸\x1b\x1b━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\x1b \x1b0.5/12.8 MB\x1b \x1b2.2 MB/s\x1b eta \x1b0:00:06\x1b',
 '\x1b[2K     \x1b━━━━\x1b\x1b╺\x1b\x1b━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\x1b \x1b1.3/12.8 MB\x1b \x1b2.8 MB/s\x1b eta \x1b0:00:05\x1b',
 '\x1b[2K     \x1b━━━━━━\x1b\x1b╸\x1b\x1b━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\x1b \x1b2.1/12.8 MB\x1b \x1b3.1 MB/s\x1b eta \x1b0:00:04\x1b',
 '\x1b[2K     \x1b━━━━━━━━━\x1b\x1b╺\x1b\x1b━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\x1b \x1b2.9/12.8 MB\x1b \x1b3.4 MB/s\x1b eta \x1b0:00:03\x1b'

In [5]:
nlp = spacy.load("en_core_web_sm")


In [6]:
def extract_who_did_what(text):
    if not isinstance(text, str):
        return ""
    
    doc = nlp(text)
    extracted_actions = []
    
    for token in doc:
        if token.pos_ == "VERB":
            subjects = [w for w in token.lefts if w.dep_ in ('nsubj', 'nsubjpass')]
            objects = [w for w in token.rights if w.dep_ in ('dobj', 'pobj', 'attr', 'dative')]
            
            if subjects and objects:
                subj_text = " ".join([t.text for t in subjects[0].subtree])
                obj_text = " ".join([t.text for t in objects[0].subtree])
                
                extracted_actions.append({
                    "Who": subj_text.strip(),
                    "Action": token.lemma_,
                    "To_What": obj_text.strip()
                })
                
    return extracted_actions

sample_text = "Ukrainian forces struck two additional logistics facilities of Russia’s largest online retailer."
print(extract_who_did_what(sample_text))

[{'Who': 'Ukrainian forces', 'Action': 'strike', 'To_What': 'two additional logistics facilities of Russia ’s largest online retailer'}]


In [7]:
df = pd.read_csv("../data/processed/scraped_with_topics_snippets_only.csv")

print("Extracting Subject-Verb-Object relationships...")
df['extracted_actions'] = df['snippet'].apply(extract_who_did_what)

actions_list = []
for index, row in df.iterrows():
    for action in row['extracted_actions']:
        actions_list.append({
            'date': row['date'],
            'Actor': action['Who'],
            'Action': action['Action'],
            'Target': action['To_What']
        })

actions_df = pd.DataFrame(actions_list)

output_path = "../data/processed/isw_extracted_actions.csv"
actions_df.to_csv(output_path, index=False)

print(f"Saved {len(actions_df)} extracted actions to {output_path}")

print("\nPreview of extracted relational data:")
print(actions_df.head(5))

Extracting Subject-Verb-Object relationships...
Saved 47 extracted actions to ../data/processed/isw_extracted_actions.csv

Preview of extracted relational data:
         date                                            Actor    Action  \
0  2026-07-23  Key Takeaways US Secretary of State Marco Rubio    reject   
1  2026-07-23                     the United States and Russia     reach   
2  2026-07-23                                 Ukrainian forces  continue   
3  2026-07-23                                   Russian forces    launch   
4  2026-07-23             Neither Russian nor Ukrainian forces      make   

                                              Target  
0  the Russian narrative that the United States a...  
1  any kind of agreement to end Russia ’s war in ...  
2  their long - range strike campaign against Rus...  
3  one Iskander - M ballistic missile , five Kh-5...  
4                                 confirmed advances  
